# TMC-LM on Google Colab

Trains the **TinyLlama-1.1B-Chat** LoRA for Trinidad Municipal College and outputs a GGUF
model for local Ollama use.

**Pipeline:** sources → dataset → LoRA fine-tune → merge → GGUF (Q4_K_M)

**Runtime:** Free tier T4 GPU. If you are on free tier, keep batch size 1 (default).

**Storage:** everything runs in this virtual machine under `/content/tmc-llm-artifacts/`.
Mounting Google Drive (cell 2) makes the model survive VM resets: each finished stage is
mirrored to `MyDrive/tmc-llm/` and can be restored on the next session (cell "2b"), so you
never re-train work you already completed.

## 1. Setup

- The repo is cloned fresh into `/content/tmc-llm` (code updates are picked up automatically).
- To train on **your own documents**, add them to `data/raw/tmc_sources/` inside the cloned repo
  (supported: `.txt`, `.md`, `.pdf`, `.docx`, `.xlsx`, `.csv`, `.json`) before running the
  dataset step. The repo ships with `train.txt` plus the full student-manual sources.
- All produced artifacts write to `/content/tmc-llm-artifacts/` in the VM:
  - `.hf-cache/` - Hugging Face model cache
  - `models/adapters`, `models/merged`, `models/gguf` - training outputs

In [ ]:
import os, pathlib, subprocess, sys

REPO_URL = "https://github.com/jbasilad/tmc-llm"
REPO_DIR = "/content/tmc-llm"

if not os.path.isdir(f"{REPO_DIR}/.git"):
    print(">> Cloning repo")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    print(">> Repo present - syncing to latest code (your untracked files are kept)")
    subprocess.run(["git", "fetch", "origin", "main"], check=True, cwd=REPO_DIR)
    subprocess.run(["git", "reset", "--hard", "origin/main"], check=True, cwd=REPO_DIR)

os.chdir(REPO_DIR)
os.environ["PYTHONPATH"] = "src"
print("Repo ready at:", os.getcwd())

## 2. Google Drive (optional, recommended)

Mounts your Drive so the trained model + zip are saved to `MyDrive/tmc-llm/` and survive VM
resets. Training still runs even if this fails - download from `/content/` at the end, or click
the **folder icon (left sidebar) → Mount Drive** and re-run this cell.

In [ ]:
import os, time

DRIVE_OK = False

def mount_drive():
    for attempt in range(1, 4):
        try:
            from google.colab import drive
            drive.mount("/content/drive", force_remount=True)
            return os.path.isdir("/content/drive/MyDrive")
        except Exception as exc:
            print(f"Drive mount attempt {attempt}/3 failed: {type(exc).__name__}: {exc}")
            time.sleep(3)
    return False

DRIVE_OK = mount_drive()

if DRIVE_OK:
    os.makedirs("/content/drive/MyDrive/tmc-llm", exist_ok=True)
    print("Google Drive mounted. Artifacts will be saved under /content/drive/MyDrive/tmc-llm/.")
    print("Tip: if this cell blocks or fails, click the folder icon -> Mount Drive instead.")
else:
    print("Drive NOT mounted. Training continues in the VM only.")
    print("Fix later: click the folder icon -> Mount Drive, then re-run the packaging cell.")

In [ ]:
ARTIFACTS = "/content/tmc-llm-artifacts"
CACHE_DIR = f"{ARTIFACTS}/.hf-cache"
SOURCES_DIR = "data/raw/tmc_sources"
ADAPTER_DIR = f"{ARTIFACTS}/models/adapters/tmc-lm-tinyllama-lora"
MERGED_DIR = f"{ARTIFACTS}/models/merged/tmc-lm-tinyllama"
GGUF_DIR = f"{ARTIFACTS}/models/gguf"

for d in [CACHE_DIR, ADAPTER_DIR, MERGED_DIR, GGUF_DIR]:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = CACHE_DIR
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

DRIVE_OK = globals().get("DRIVE_OK", False)

print("Artifact paths:")
print("  HF cache :", CACHE_DIR)
print("  Sources  :", SOURCES_DIR)
print("  Adapters :", ADAPTER_DIR)
print("  Merged   :", MERGED_DIR)
print("  GGUF     :", GGUF_DIR)
print()
print("Google Drive persistence:", "ON (mirrored to MyDrive/tmc-llm as stages finish)" if DRIVE_OK else "OFF  (VM-only, download from /content/)")

**2b. Resume from Google Drive (optional)**

If you trained before and the VM was reset, this cell copies the adapter / GGUF back from Drive
so you never re-train completed work. If the VM already has newer output it is **not** overwritten.

> Run this once after mounting Drive. It is safe to re-run.

In [ ]:
import glob as _g, os as _os, shutil as _shutil

DRIVE_ROOT = "/content/drive/MyDrive/tmc-llm"
DRIVE_OK = globals().get("DRIVE_OK", False)

def copy_tree_contents(src, dest_root):
    if not _os.path.isdir(src):
        return 0
    _os.makedirs(dest_root, exist_ok=True)
    n = 0
    for root, _dirs, names in _os.walk(src):
        rel = _os.path.relpath(root, src)
        target = _os.path.join(dest_root, rel if rel != "." else "")
        _os.makedirs(target, exist_ok=True)
        for name in names:
            _shutil.copy2(_os.path.join(root, name), _os.path.join(target, name))
            n += 1
    return n

def vm_has_output(vm_dir, min_mb):
    matches = _g.glob(f"{vm_dir}/**/*", recursive=True)
    return any(_os.path.isfile(f) and _os.path.getsize(f) >= min_mb * 1e6 for f in matches)

restored_any = False

for drive_dir, vm_dir, label, min_mb in [
    (f"{DRIVE_ROOT}/artifacts/adapters", ADAPTER_DIR, "LoRA adapter", 1),
    (f"{DRIVE_ROOT}/artifacts/gguf", GGUF_DIR, "GGUF model", 100),
]:
    if DRIVE_OK and _os.path.isdir(drive_dir):
        if not vm_has_output(vm_dir, min_mb):
            n = copy_tree_contents(drive_dir, vm_dir)
            if n:
                print(f"[Restore] {label}: {n} file(s) copied from Drive -> {vm_dir}")
                restored_any = True
        else:
            print(f"[Restore] {label}: VM already has output, keeping it.")

if DRIVE_OK and restored_any:
    print("Restored previous training from Drive - you can run just the packaging cell now,")
    print("or Run all again to re-train on top of the restored adapter.")
elif DRIVE_OK:
    print("No previous artifacts on Drive yet - will train from scratch (that is normal the first time).")
else:
    print("Drive not mounted - no restore possible. (Mount Drive, then re-run Drive + this cell.)")

In [ ]:
import glob, json, os, pathlib, subprocess, sys

def run_script(script):
    print(f">> Running {script}")
    log = "/content/run_script.log"
    cmd = f"/bin/bash {script} 2>&1 | tee {log}"
    proc = subprocess.Popen(cmd, shell=True, text=True)
    rc = proc.wait()
    if rc != 0:
        print(f"\n=== {script} FAILED (exit {rc}). Last 40 log lines: ===")
        try:
            lines = open(log, encoding="utf-8", errors="replace").read().splitlines()
            print("\n".join(lines[-40:]))
        except Exception as exc:
            print(f"(could not read log: {exc})")
        raise RuntimeError(f"{script} exited with code {rc} -- see log above")

print(">> Removing Colab's preinstalled torchao (its old version breaks peft LoRA)")
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)

print(">> Installing ML libraries (torch already present in Colab)")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "peft", "datasets", "accelerate", "sentencepiece",
    "PyMuPDF", "python-docx", "openpyxl", "pyyaml", "gguf",
], check=True)

print(">> Installing the tmc_llm package")
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    print("   tmc_llm installed as editable package")
except subprocess.CalledProcessError as exc:
    print("   editable install failed, continuing with PYTHONPATH=src")
    print("   (pip exit", exc.returncode, ")")

import torch
device = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print("PyTorch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| Device:", device)

# --- per-stage verification helpers -------------------------------------------
STATUS_FILE = f"{ARTIFACTS}/pipeline-status.json"

def save_status(**stages):
    status = {}
    try:
        status = json.load(open(STATUS_FILE, encoding="utf-8"))
    except Exception:
        pass
    status.update(stages)
    with open(STATUS_FILE, "w", encoding="utf-8") as fh:
        json.dump(status, fh, indent=2)

def log_tail():
    try:
        lines = open("/content/run_script.log", encoding="utf-8", errors="replace").read().splitlines()
        return "\n".join(lines[-40:])
    except Exception as exc:
        return f"(no log available: {exc})"

def verify_stage(name, *requirements):
    # requirements: (glob_pattern, min_mb) tuples.
    matched = []
    lines = []
    ok = True
    for pat, min_mb in requirements:
        hits = [p for p in glob.glob(pat, recursive=True)
                if os.path.isfile(p) and os.path.getsize(p) >= min_mb * 1e6]
        matched.extend(hits)
        if hits:
            lines.append(f"[OK] {pat!r} (>= {min_mb:.2f} MB) -> {len(hits)} file(s)")
        else:
            ok = False
            lines.append(f"[MISSING] {pat!r} (>= {min_mb:.2f} MB)")
    if not ok:
        print("=" * 72)
        print(f"STAGE FAILED: '{name}' did not produce the expected outputs.")
        print("  checks:")
        for line in lines:
            print("   ", line)
        seen = set()
        for pat, _ in requirements:
            d = os.path.dirname(pat)
            if d in seen or not os.path.isdir(d):
                continue
            seen.add(d)
            print(f"  listing {d}:")
            for f in sorted(glob.glob(f"{d}/**/*", recursive=True)):
                if os.path.isfile(f):
                    print(f"    {os.path.getsize(f)/1e6:9.1f} MB  {f}")
        print("  last 40 log lines (run_script.log):")
        print(log_tail())
        print("=" * 72)
        raise RuntimeError(f"Stage '{name}' produced no valid output - see messages above.")
    save_status(**{name: {"ok": True, "patterns": [p for p, _ in requirements]}})
    print(f"[OK] {name}")
    for line in lines:
        print("   ", line)
    return matched

## 3. Prepare the dataset

Builds `data/processed/{dataset,train,validation,test}.jsonl` from the source documents in
`data/raw/tmc_sources/`.

In [ ]:
os.environ["SOURCE_DIR"] = SOURCES_DIR
os.environ["OUTPUT_DIR"] = "data/processed"
run_script("scripts/prepare_dataset.sh")
verify_stage("dataset", ("data/processed/metadata.json", 0), ("data/processed/dataset.jsonl", 0.01))
print(">> Dataset ready")

In [ ]:
import json
metadata = json.load(open("data/processed/metadata.json", encoding="utf-8"))
print(f"Source files   : {len(metadata['source_files'])}")
print(f"Total examples : {metadata['total_examples']}")
print(f"Train examples : {metadata['train_examples']}")
print(f"Validation     : {metadata['validation_examples']}")
print(f"Test examples  : {metadata['test_examples']}")
print("Sections       :", ", ".join(metadata["sections"]))

## 4. Fine-tune TinyLlama with LoRA

Uses `configs/train_lora.yaml` hyperparameters (fp16, batch size 1, ~80 steps) but sends the
adapter to `/content/tmc-llm-artifacts/models/adapters/`. On a free T4 this takes roughly
20-40 minutes (plus the one-time ~2.2 GB model download on first run). Once it finishes, the
adapter is mirrored to Drive (if mounted).

In [ ]:
import pathlib
import yaml

cfg = yaml.safe_load(pathlib.Path("configs/train_lora.yaml").read_text(encoding="utf-8"))
cfg["output_dir"] = ADAPTER_DIR

colab_cfg = pathlib.Path("configs/train_lora_colab.yaml")
colab_cfg.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
print(colab_cfg.read_text(encoding="utf-8"))

In [ ]:
os.environ["CONFIG_PATH"] = "configs/train_lora_colab.yaml"
run_script("scripts/train_lora.sh")
verify_stage("train", (f"{ADAPTER_DIR}/**/adapter_model.*", 1))
print(">> Adapter saved to", ADAPTER_DIR)

DRIVE_OK = globals().get("DRIVE_OK", False)
if DRIVE_OK:
    n = copy_tree_contents(ADAPTER_DIR, f"{DRIVE_ROOT}/artifacts/adapters")
    print(f"[Drive] adapter checkpoint: {n} file(s) -> {DRIVE_ROOT}/artifacts/adapters/")

## 5. Merge the LoRA adapter into the base model

Produces a full merged model at `/content/tmc-llm-artifacts/models/merged/tmc-lm-tinyllama/`.

In [ ]:
os.environ["BASE_MODEL"] = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
os.environ["ADAPTER_DIR"] = ADAPTER_DIR
os.environ["OUTPUT_DIR"] = MERGED_DIR
run_script("scripts/merge_lora.sh")
verify_stage("merge", (f"{MERGED_DIR}/model.safetensors", 50), (f"{MERGED_DIR}/config.json", 0))
print(">> Merged model saved to", MERGED_DIR)

## 6. Convert to GGUF (llama.cpp, built inside Colab)

No Docker needed. The script clones llama.cpp, builds only the `llama-quantize` binary
(CPU-only, portable), converts the merged model to F16, then quantizes to **Q4_K_M**.
First run takes a few extra minutes for the llama.cpp build.

In [ ]:
os.environ["MERGED_MODEL_DIR"] = MERGED_DIR
os.environ["OUTPUT_DIR"] = GGUF_DIR
os.environ["BUILD_JOBS"] = "2"
run_script("scripts/convert_to_gguf.sh")

In [ ]:
import glob, os

gguf_files = verify_stage("convert", (f"{GGUF_DIR}/*.gguf", 100))
for f in sorted(gguf_files):
    print(f"{os.path.basename(f):<40} {os.path.getsize(f)/1e6:8.1f} MB")

DRIVE_OK = globals().get("DRIVE_OK", False)
if DRIVE_OK:
    n = copy_tree_contents(GGUF_DIR, f"{DRIVE_ROOT}/artifacts/gguf")
    print(f"[Drive] GGUF checkpoint: {n} file(s) -> {DRIVE_ROOT}/artifacts/gguf/")

## 7. Sanity check (in-notebook QA)

Quick test of the **merged** model against a few TMC knowledge questions before you download.

In [ ]:
import json, pathlib, torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

merged_path = pathlib.Path(MERGED_DIR)
print("Merged dir:", merged_path)
if merged_path.exists():
    print("  contents:", sorted(p.name for p in merged_path.iterdir()))
else:
    print("  !! MISSING merged model directory")

cfg_path = merged_path / "config.json"
if cfg_path.exists():
    cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
    print("  config.json model_type:", cfg.get("model_type"))
    if not cfg.get("model_type"):
        print("  >> repairing config.json (missing model_type)")
        base = json.loads(AutoConfig.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0").to_json_string())
        base.update({k: v for k, v in cfg.items() if k != "model_type"})
        cfg_path.write_text(json.dumps(base, indent=2), encoding="utf-8")
        print("  >> repaired")
else:
    print("  !! config.json missing - writing from base model")
    AutoConfig.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0").save_pretrained(MERGED_DIR)

try:
    qa_model = AutoModelForCausalLM.from_pretrained(MERGED_DIR, torch_dtype=torch.float16, device_map="auto")
    qa_tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)

    SYSTEM = (
        "You are TMC-LM, an offline assistant for Trinidad Municipal College. "
        "Answer using only official TMC knowledge. If the answer is not in the "
        "source, say that the available TMC source does not contain it. "
        "Be concise and professional."
    )

    def format_for_qa(tokenizer, messages):
        if getattr(tokenizer, "chat_template", None):
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        user = next(m["content"] for m in messages if m["role"] == "user")
        return f"### User: {user}\n### Assistant:"

    for question in [
        "What is the vision of TMC?",
        "What academic programs does TMC offer?",
        "Give a brief history of TMC.",
    ]:
        prompt = format_for_qa(
            qa_tokenizer,
            [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}],
        )
        inputs = qa_tokenizer(prompt, return_tensors="pt").to(qa_model.device)
        outputs = qa_model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.2,
            top_p=0.9,
        )
        answer = qa_tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        print(f"\nQ: {question}\nA: {answer.strip()}\n")
except Exception as exc:
    print(f"\nWARNING: sanity check failed ({type(exc).__name__}: {exc}) - continuing.")
    print("Your GGUF model is still ready for download below.")

## 8. Download your model

Everything lives in the VM under `/content/tmc-llm-artifacts/`. The cell below:

- **Verifies** the GGUF and LoRA adapter actually exist (no more empty downloads);
- Falls back to **Drive's copy** if the VM was freshly reset;
- Zips into `/content/tmc-llm-download.zip`;
- With Drive mounted, mirrors everything to `MyDrive/tmc-llm/` so it survives VM resets.

Main files:
- `/content/tmc-llm-artifacts/models/gguf/tmc-lm-tinyllama-q4_k_m.gguf` - **for Ollama**
- `/content/tmc-llm-artifacts/models/adapters/` - LoRA adapter (optional backup)

Then import into local Ollama on Windows: `scripts/create_ollama_model.ps1` + `ollama run tmc-lm`.

> **One-sitting rule:** the trained model only exists in the VM that trained it. After any
> disconnect/restart, `/content` is wiped - either run everything in one sitting and download
> before closing, or rely on Drive (cell 2 + 2b) to keep/resume your work. If a stage aborts
> early, its `STAGE FAILED` message tells you exactly which step broke - paste it for help.

In [ ]:
import glob, os, pathlib, shutil

stage = "/content/tmc-llm-download"
zip_path = "/content/tmc-llm-download.zip"

def mb(path):
    return os.path.getsize(path) / 1e6

DRIVE_ROOT = "/content/drive/MyDrive/tmc-llm"
DRIVE_OK = globals().get("DRIVE_OK", False)

ggufs = sorted(glob.glob(f"{GGUF_DIR}/*.gguf"))
adapters = sorted(glob.glob(f"{ADAPTER_DIR}/**/adapter_model.*", recursive=True))
bad = [p for p in ggufs if mb(p) < 100] + [p for p in adapters if mb(p) < 1]

if (not ggufs or not adapters or bad) and DRIVE_OK:
    print("VM artifacts missing/invalid - trying the Drive copy before giving up ...")
    restored = copy_tree_contents(f"{DRIVE_ROOT}/artifacts/gguf", GGUF_DIR)
    restored += copy_tree_contents(f"{DRIVE_ROOT}/artifacts/adapters", ADAPTER_DIR)
    print(f"[Restore] {restored} file(s) copied from Drive into the VM")
    ggufs = sorted(glob.glob(f"{GGUF_DIR}/*.gguf"))
    adapters = sorted(glob.glob(f"{ADAPTER_DIR}/**/adapter_model.*", recursive=True))
    bad = [p for p in ggufs if mb(p) < 100] + [p for p in adapters if mb(p) < 1]

if not ggufs or not adapters or bad:
    print("=" * 72)
    print("NO VALID TRAINED MODEL FOUND - packaging ABORTED (no empty zip created).")
    print(f"  GGUF files   : {len(ggufs)}  {[os.path.basename(f) for f in ggufs]}")
    print(f"  Adapter files : {len(adapters)}  {[os.path.basename(f) for f in adapters]}")
    if bad:
        print(f"  Too small    : {[os.path.basename(f) for f in bad]}")
    print()
    print("  The artifact folders are empty or incomplete. This usually means the VM")
    print("  was reset (or a training stage failed earlier) and nothing was trained.")
    print()
    print("  Fix: Runtime -> Run all (restart) and wait for the ENTIRE pipeline to finish.")
    print("  If a stage aborts early, its 'STAGE FAILED' message shows the exact step.")
    print("=" * 72)
    raise RuntimeError("Missing/invalid trained model - refusing to produce an empty zip.")

pathlib.Path(stage).mkdir(parents=True, exist_ok=True)
shutil.copytree(GGUF_DIR, f"{stage}/gguf", dirs_exist_ok=True)
shutil.copytree(ADAPTER_DIR, f"{stage}/adapters", dirs_exist_ok=True)
shutil.make_archive(zip_path[:-4], "zip", root_dir=stage)
print("VM zip created:", zip_path, f"({mb(zip_path):.1f} MB)")

if DRIVE_OK:
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    copy_tree_contents(GGUF_DIR, f"{DRIVE_ROOT}/artifacts/gguf")
    copy_tree_contents(ADAPTER_DIR, f"{DRIVE_ROOT}/artifacts/adapters")
    shutil.copy2(zip_path, f"{DRIVE_ROOT}/tmc-llm-download.zip")
    print()
    print("Saved to Google Drive:")
    print("  ", f"{DRIVE_ROOT}/tmc-llm-download.zip", f"({mb(zip_path):.1f} MB)")
    print("  ", f"{DRIVE_ROOT}/artifacts/gguf/")
    print("  ", f"{DRIVE_ROOT}/artifacts/adapters/")
else:
    print()
    print("Drive NOT mounted - skipped Drive copy. Use the /content/ zip instead.")
    print("(Or: folder icon -> Mount Drive, then re-run this cell and it will re-copy.)")

print()
print("GGUF files in package:")
for f in ggufs:
    print("  ", os.path.basename(f), f"({mb(f):.1f} MB)")

print()
print("Download / import:")
print("  1. File browser -> /content/tmc-llm-download.zip -> Download  (or grab the copy on Drive)")
print("  2. On Windows in the tmc-llm repo: put the GGUF in  models/gguf/")
print("  3.    scripts/check_gguf.ps1   (optional)")
print("  4.    scripts/create_ollama_model.ps1")
print("  5.    ollama run tmc-lm")